In [ ]:
# CELL 1 - Install dependencies
!pip install torch diffusers transformers accelerate
!pip install fastapi uvicorn pyngrok
!pip install opencv-python pillow numpy huggingface_hub

In [ ]:
# CELL 2 - Download Wan2.1 model
import os
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except:
    hf_token = os.environ.get("HF_TOKEN", "[HF_TOKEN]")

print("Downloading Wan2.1 model...")
model_id = "Wan-AI/Wan2.1-T2V-1.3B"
local_dir = "/kaggle/working/wan-model"
if hf_token and hf_token != "[HF_TOKEN]":
    snapshot_download(repo_id=model_id, local_dir=local_dir, token=hf_token)
else:
    snapshot_download(repo_id=model_id, local_dir=local_dir)
print("Model downloaded successfully!")

In [ ]:
# CELL 3 - Start FastAPI server with 5-minute idle timeout
import os
import threading
import uvicorn
import time
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

last_activity = time.time()
IDLE_TIMEOUT = 5 * 60  # 5 minutes

class GenerateRequest(BaseModel):
    prompt: str
    scene_id: str
    account_id: str

@app.post("/generate")
def generate_video(request: GenerateRequest):
    global last_activity
    last_activity = time.time()
    print(f"Received generation request: {request.prompt} for scene {request.scene_id}")
    time.sleep(10)
    output_file = f"/kaggle/working/{request.scene_id}.mp4"
    with open(output_file, 'wb') as f:
        f.write(b"Dummy video content")
    last_activity = time.time()
    return {"status": "success", "video_path": output_file}

@app.get("/health")
def health():
    return {"status": "alive"}

def idle_checker():
    global last_activity
    while True:
        time.sleep(30)
        idle_secs = time.time() - last_activity
        print(f"Idle for {int(idle_secs)}s / {IDLE_TIMEOUT}s")
        if idle_secs > IDLE_TIMEOUT:
            print("Idle timeout reached (5 mins). Shutting down to save GPU quota...")
            os._exit(0)

idle_thread = threading.Thread(target=idle_checker, daemon=True)
idle_thread.start()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("FastAPI server started on port 8000")

In [ ]:
# CELL 4 - Start ngrok tunnel & Register with backend
import os
import requests
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    ngrok_token = user_secrets.get_secret("NGROK_TOKEN")
except:
    ngrok_token = os.environ.get("NGROK_TOKEN", "[NGROK_TOKEN]")
    
try:
    user_secrets = UserSecretsClient()
    kaggle_user = user_secrets.get_secret("KAGGLE_USERNAME")
except:
    kaggle_user = os.environ.get("KAGGLE_USERNAME", "Unknown")

if ngrok_token and ngrok_token != "[NGROK_TOKEN]":
    ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8000).public_url
print(f"Ngrok Tunnel URL: {public_url}")

# Register to Modal Backend
backend_url = "https://irfangull2288--cartoon-backend-fastapi-modal-app.modal.run/session/register-url"
try:
    resp = requests.post(backend_url, json={
        "account_username": kaggle_user,
        "url": public_url
    })
    print("Backend registration response:", resp.json())
except Exception as e:
    print("Failed to register with backend:", e)

In [ ]:
# CELL 5 - Keep alive (idle checker will auto-shutdown after 5 min of no requests)
import time
print("Ready! Will auto-shutdown after 5 minutes of idle.")
while True:
    time.sleep(60)